# Gymnasium — The Standard Interface for Reinforcement Learning Environments

---

## What Is This Notebook About?

This notebook teaches you **Gymnasium** (formerly OpenAI Gym), the universal API that every reinforcement learning library speaks. Think of it as the USB standard — just like every USB device works with any USB port regardless of brand, every RL algorithm works with any Gymnasium environment regardless of what the game or simulation is.

By the end of this notebook you will understand:
- What reinforcement learning (RL) actually is and why it matters
- The Gymnasium API: `make`, `reset`, `step`, `render`
- Observation spaces, action spaces, rewards, and episodes
- How to build your own custom environment from scratch
- Wrappers for modifying environments
- A complete mini-project: Grid World navigation agent

---

## Real-World Analogy: Teaching a Dog New Tricks

Imagine you're training a dog to fetch a ball:
- **Environment**: The park (where the dog lives and acts)
- **Agent**: The dog (the one making decisions)
- **State/Observation**: What the dog sees — where the ball is, where you are
- **Action**: What the dog does — run left, run right, pick up ball
- **Reward**: A treat (+1) when it fetches the ball, nothing (0) otherwise
- **Episode**: One complete fetch attempt (start → fetch → end)

The dog learns over thousands of tries that certain actions in certain situations lead to treats. That's **Reinforcement Learning** — learning through trial, error, and rewards.

Gymnasium is the **park** — the standardized environment where training happens.

---

## Why Does This Matter?

| Application | RL Agent | What It Learned |
|-------------|----------|------------------|
| AlphaGo (DeepMind) | Go player | Beat world champion |
| ChatGPT RLHF | Language model | Be helpful & harmless |
| Robot arm | Robot | Grasp and move objects |
| Trading bot | Trader | Buy/sell decisions |
| Self-driving car | Car | Navigate roads safely |
| Game AI | Player | Beat Atari games at superhuman level |

All of these were built and tested using environments similar to Gymnasium's interface.

---

## Prerequisites
- Python basics (loops, functions, classes)
- NumPy arrays
- No prior RL knowledge needed!

---

## Table of Contents
1. Installation & Setup
2. The RL Loop Explained
3. Core Spaces: Observation & Action
4. Classic Environments
5. Implementing a Random Agent
6. Environment Wrappers
7. Building a Custom Environment
8. Monitoring Training Progress
9. Common Pitfalls
10. Mini Project: Grid World
11. Interview Q&A
12. Resources

---

## Official Resources
- **Docs**: https://gymnasium.farama.org/
- **GitHub**: https://github.com/Farama-Foundation/Gymnasium
- **YouTube Tutorial (Nicholas Renotte)**: https://www.youtube.com/watch?v=bD6V3rcr_54
- **Spinning Up in RL (OpenAI)**: https://spinningup.openai.com/en/latest/
- **Sutton & Barto (free RL textbook)**: http://incompleteideas.net/book/the-book-2nd.html

## 1. Installation & Setup

In [ ]:
# Install command (run in terminal):
# pip install gymnasium
# pip install gymnasium[classic-control]  # For CartPole, MountainCar, etc.
# pip install gymnasium[atari]            # For Atari games (large download)
# pip install matplotlib numpy

try:
    import gymnasium as gym
    GYM_AVAILABLE = True
    print(f"Gymnasium version: {gym.__version__}")
except ImportError:
    GYM_AVAILABLE = False
    print("Gymnasium not installed. Run: pip install gymnasium[classic-control]")
    print("All code blocks will simulate output for learning purposes.")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

print("NumPy version:", np.__version__)
print("Setup complete!")

## 2. The RL Loop Explained

Every Reinforcement Learning system follows the same loop:

```
┌─────────────────────────────────────────────────────┐
│                   ENVIRONMENT                        │
│                                                     │
│   state(t)  ──────────────────────►  Agent          │
│                                       │             │
│   reward(t) ──────────────────────►   │             │
│                                       │ decides     │
│   next_state(t+1) ◄── env.step() ◄──  action(t)    │
│   reward(t+1)                                       │
│   done                                              │
└─────────────────────────────────────────────────────┘
```

In code, this is always:
```python
env = gym.make('SomeGame-v1')          # Create the world
observation, info = env.reset()         # Start fresh, get initial state

while not done:
    action = agent.choose(observation)  # Agent decides what to do
    observation, reward, terminated, truncated, info = env.step(action)  # World responds
    done = terminated or truncated      # Is the episode over?
```

### The 5 Return Values from `env.step(action)`:
| Value | Type | Meaning |
|-------|------|---------|
| `observation` | numpy array | What the agent sees next |
| `reward` | float | How good was the action? |
| `terminated` | bool | Did the agent WIN or LOSE? (natural end) |
| `truncated` | bool | Did the episode hit a time limit? (timeout) |
| `info` | dict | Extra debugging info (distance, lives, etc.) |

> **Gymnasium vs old Gym**: OpenAI's Gym used to return 4 values: `obs, reward, done, info`. The new Gymnasium splits `done` into `terminated` and `truncated`. This distinction matters: termination means the agent reached a goal/failure state; truncation means just a time limit.

In [ ]:
# ── The Complete RL Loop ──────────────────────────────────────────

if GYM_AVAILABLE:
    env = gym.make('CartPole-v1')  # Famous balancing pole problem

    # Reset: always call this at the start of each episode
    observation, info = env.reset(seed=42)

    print("=== CartPole-v1 Environment ===")
    print(f"Initial observation: {observation}")
    print(f"  [cart_position, cart_velocity, pole_angle, pole_angular_velocity]")
    print(f"Info: {info}")
    print()

    # Run one complete episode with random actions
    total_reward = 0
    step = 0

    for step in range(200):  # Max 200 steps in CartPole-v1
        # Random action: 0 = push cart LEFT, 1 = push cart RIGHT
        action = env.action_space.sample()

        # Execute action, get results
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        if terminated or truncated:
            break

    env.close()
    print(f"Episode finished after {step+1} steps")
    print(f"Total reward: {total_reward}")
    print(f"(A perfect CartPole agent scores 500 — the max!)")

else:
    # Simulated output for learning
    print("=== CartPole-v1 Environment ===")
    print("Initial observation: [ 0.0271  -0.0444  -0.0054  -0.0258]")
    print("  [cart_position, cart_velocity, pole_angle, pole_angular_velocity]")
    print("Info: {}")
    print()
    print("Episode finished after 23 steps")
    print("Total reward: 23.0")
    print("(A perfect CartPole agent scores 500 — the max!)")

## 3. Core Spaces: Observation Space & Action Space

Every Gymnasium environment has two critical attributes:

### Observation Space
Describes **what the agent can see**. For CartPole, it's 4 numbers. For Atari, it's a 210×160×3 image.

### Action Space
Describes **what the agent can do**.

| Space Type | Description | Example |
|------------|-------------|--------|
| `Discrete(n)` | n possible actions (integers 0 to n-1) | CartPole: 2 (left/right) |
| `Box(low, high, shape)` | Continuous values in a range | Robot arm: joint torques |
| `MultiBinary(n)` | n binary (on/off) switches | Which items to carry |
| `MultiDiscrete([n1, n2])` | Multiple discrete choices | (attack, move_direction) |
| `Dict({'a': Discrete(3), 'b': Box(...)})` | Complex structured spaces | Observations with metadata |

Understanding spaces is critical because:
1. Your neural network input size = observation space shape
2. Your neural network output size = action space size (for discrete) or range (for continuous)

In [ ]:
# ── Exploring Spaces ──────────────────────────────────────────────

if GYM_AVAILABLE:
    environments_to_explore = [
        ('CartPole-v1', 'Discrete'),
        ('MountainCarContinuous-v0', 'Continuous'),
        ('LunarLander-v2', 'Discrete'),
    ]

    for env_name, space_type in environments_to_explore:
        try:
            env = gym.make(env_name)
            print(f"{'='*50}")
            print(f"Environment: {env_name} ({space_type} actions)")
            print(f"  Observation space: {env.observation_space}")
            print(f"  Observation shape: {env.observation_space.shape}")
            print(f"  Observation dtype: {env.observation_space.dtype}")
            print(f"  Action space:      {env.action_space}")

            if hasattr(env.action_space, 'n'):
                print(f"  Number of actions: {env.action_space.n}")
            elif hasattr(env.action_space, 'shape'):
                print(f"  Action shape:  {env.action_space.shape}")
                print(f"  Action low:    {env.action_space.low}")
                print(f"  Action high:   {env.action_space.high}")

            # Sample random observations and actions
            random_obs = env.observation_space.sample()
            random_act = env.action_space.sample()
            print(f"  Sample obs:    {random_obs[:4] if len(random_obs) > 4 else random_obs}")
            print(f"  Sample action: {random_act}")
            env.close()
        except Exception as e:
            print(f"  Could not load {env_name}: {e}")
else:
    print("=" * 50)
    print("Environment: CartPole-v1 (Discrete actions)")
    print("  Observation space: Box(-4.8, 4.8, (4,), float32)")
    print("  Observation shape: (4,)")
    print("  Action space:      Discrete(2)")
    print("  Number of actions: 2  (0=Left, 1=Right)")
    print("  Sample obs:    [ 0.038  0.012 -0.044  0.019]")
    print("  Sample action: 1")
    print("="*50)
    print("Environment: MountainCarContinuous-v0 (Continuous actions)")
    print("  Observation space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)")
    print("  Action space:      Box([-1.], [1.], (1,), float32)")
    print("  Action shape:  (1,)  (how much to push: -1.0 to +1.0)")
    print("="*50)
    print("Environment: LunarLander-v2 (Discrete actions)")
    print("  Observation space: Box(-inf, inf, (8,), float32)")
    print("  Action space:      Discrete(4)")
    print("  Actions: 0=Do nothing, 1=Fire left, 2=Fire main, 3=Fire right")

## 4. Classic Environments — The Standard Benchmarks

These environments are used everywhere in RL research to benchmark algorithms.

| Environment | Task | Obs Space | Action Space | Solved When |
|-------------|------|-----------|--------------|-------------|
| `CartPole-v1` | Balance a pole | 4 floats | Discrete(2) | Avg reward ≥ 475 |
| `MountainCar-v0` | Drive up hill | 2 floats | Discrete(3) | Avg reward ≥ -110 |
| `LunarLander-v2` | Land rocket | 8 floats | Discrete(4) | Avg reward ≥ 200 |
| `Pendulum-v1` | Swing up pendulum | 3 floats | Box(1) | Min avg reward |
| `Acrobot-v1` | Swing up 2-link arm | 6 floats | Discrete(3) | Avg reward ≥ -100 |
| `Breakout-v4` | Atari breakout | (210,160,3) | Discrete(4) | High score |

### CartPole Explained
A cart moves on a track. A pole is attached to the cart. The goal: keep the pole upright by pushing the cart left or right.
- Observation: [cart_x, cart_velocity, pole_angle, pole_angular_velocity]
- Actions: 0 (push left), 1 (push right)
- Reward: +1 for every step the pole stays up
- Episode ends: pole falls (>15°) or cart leaves track, or 500 steps

This is RL's equivalent of "Hello World".

In [ ]:
# ── Visualizing Multiple Episodes ────────────────────────────────────

def run_episode_random(env_name, max_steps=500, seed=None):
    """Run one episode with a random agent, return total reward."""
    if GYM_AVAILABLE:
        env = gym.make(env_name)
        obs, _ = env.reset(seed=seed)
        total_reward = 0
        rewards_per_step = []
        for _ in range(max_steps):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            rewards_per_step.append(reward)
            if terminated or truncated:
                break
        env.close()
        return total_reward, len(rewards_per_step)
    else:
        # Simulate random scores
        steps = np.random.randint(10, 50)
        return float(steps), steps


# Run 20 episodes and plot the distribution of scores
n_episodes = 20
scores = []
lengths = []

for ep in range(n_episodes):
    score, length = run_episode_random('CartPole-v1', seed=ep) if GYM_AVAILABLE else (np.random.randint(10,50), np.random.randint(10,50))
    scores.append(score)
    lengths.append(length)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Episode scores
axes[0].plot(scores, 'bo-', alpha=0.7, linewidth=1.5)
axes[0].axhline(np.mean(scores), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(scores):.1f}')
axes[0].fill_between(range(n_episodes), np.mean(scores)-np.std(scores), np.mean(scores)+np.std(scores), alpha=0.2, color='red')
axes[0].set_title('Random Agent: CartPole-v1 Scores', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward (= # Steps Survived)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].text(0.05, 0.95, 'Max possible: 500\nRandom avg: ~20-25', transform=axes[0].transAxes,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Score distribution
axes[1].hist(scores, bins=10, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Score Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Frequency')
axes[1].axvline(np.mean(scores), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(scores):.1f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Random Policy Performance on CartPole-v1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/gymnasium_random_agent.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nRandom agent performance over {n_episodes} episodes:")
print(f"  Mean score:  {np.mean(scores):.1f}")
print(f"  Std:         {np.std(scores):.1f}")
print(f"  Min / Max:   {min(scores):.0f} / {max(scores):.0f}")
print(f"  Solved criterion: avg >= 475 over 100 episodes")

## 5. Implementing a Simple Rule-Based Agent

Before using RL algorithms, let's understand the problem better by writing a **rule-based agent** — one that uses human logic instead of learned policy.

For CartPole, the pole angle is the most important signal. Intuition:
- If the pole is tilting **right** (angle > 0) → push cart **right** (action=1) to compensate
- If the pole is tilting **left** (angle < 0) → push cart **left** (action=0) to compensate

In [ ]:
# ── Rule-Based Agent vs Random Agent ─────────────────────────────────

class RandomAgent:
    """Selects actions completely at random — our baseline."""
    def __init__(self, action_space):
        self.action_space = action_space

    def act(self, observation):
        return self.action_space.sample()


class CartPoleRuleAgent:
    """
    Hand-coded rule: push toward the direction the pole is falling.
    Observation indices for CartPole-v1:
        0: cart position
        1: cart velocity
        2: pole angle       <-- most important!
        3: pole angular velocity
    """
    def act(self, observation):
        pole_angle = observation[2]
        pole_velocity = observation[3]
        # Use both angle and velocity for better control
        if (pole_angle + 0.1 * pole_velocity) > 0:
            return 1  # Push right
        else:
            return 0  # Push left


def evaluate_agent(agent, env_name, n_episodes=20, max_steps=500):
    """Evaluate an agent over n episodes."""
    scores = []
    if GYM_AVAILABLE:
        env = gym.make(env_name)
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=ep)
            total_reward = 0
            for _ in range(max_steps):
                if hasattr(agent, 'action_space'):
                    action = agent.act(obs)  # RandomAgent needs action_space internally
                else:
                    action = agent.act(obs)
                obs, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                if terminated or truncated:
                    break
            scores.append(total_reward)
        env.close()
    else:
        # Simulated scores
        if isinstance(agent, CartPoleRuleAgent):
            scores = [np.random.randint(200, 500) for _ in range(n_episodes)]
        else:
            scores = [np.random.randint(8, 45) for _ in range(n_episodes)]
    return scores


# Evaluate both agents
if GYM_AVAILABLE:
    env_dummy = gym.make('CartPole-v1')
    random_agent = RandomAgent(env_dummy.action_space)
    env_dummy.close()
else:
    random_agent = RandomAgent(None)

rule_agent = CartPoleRuleAgent()

random_scores = evaluate_agent(random_agent, 'CartPole-v1', n_episodes=20)
rule_scores   = evaluate_agent(rule_agent, 'CartPole-v1', n_episodes=20)

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 5))

episodes = range(20)
ax.plot(episodes, random_scores, 'ro-', alpha=0.7, label=f'Random (mean={np.mean(random_scores):.0f})', linewidth=1.5)
ax.plot(episodes, rule_scores, 'go-', alpha=0.7, label=f'Rule-Based (mean={np.mean(rule_scores):.0f})', linewidth=1.5)
ax.axhline(475, color='gold', linestyle='--', linewidth=2, label='Solved threshold (475)')
ax.set_title('Random vs Rule-Based Agent on CartPole-v1', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/gymnasium_agents_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nRandom Agent:     mean={np.mean(random_scores):.1f}, max={max(random_scores):.0f}")
print(f"Rule-Based Agent: mean={np.mean(rule_scores):.1f}, max={max(rule_scores):.0f}")
print(f"\nThe rule-based agent is better, but it's still hand-coded.")
print("RL agents learn these rules AUTOMATICALLY through trial and error!")

## 6. Environment Wrappers

Wrappers let you **modify an environment without changing its source code**. Think of it like a phone case — it goes around the phone and adds features (protection, a stand, card slots) without modifying the phone itself.

Common wrapper use cases:
- **Scale rewards**: normalize rewards to [-1, +1] for stable training
- **Stack frames**: for Atari, stack 4 frames so the agent sees motion
- **Record video**: save videos of your agent playing
- **Time limit**: add maximum episode length
- **Normalize observations**: zero-mean, unit-variance observations

Built-in Gymnasium wrappers:
```python
gym.wrappers.RecordVideo(env, 'video_folder')     # Record video
gym.wrappers.TimeLimit(env, max_episode_steps=100) # Cap episodes
gym.wrappers.ClipReward(env, min=-1, max=1)        # Clip rewards
gym.wrappers.NormalizeObservation(env)             # Normalize obs
gym.wrappers.FrameStack(env, num_stack=4)          # Stack frames
```

In [ ]:
# ── Building a Custom Wrapper ─────────────────────────────────────────

# Example: A wrapper that logs every reward and prints a summary

if GYM_AVAILABLE:
    class RewardTrackingWrapper(gym.Wrapper):
        """
        Wraps any Gymnasium environment to track rewards.
        Like a fitness tracker wrapped around your wrist — the wrist
        doesn't change, but now you can see your heart rate!
        """
        def __init__(self, env, reward_scale=1.0):
            super().__init__(env)   # Always call super().__init__(env)
            self.reward_scale = reward_scale
            self.episode_rewards = []
            self._current_episode_reward = 0
            self._episode_count = 0

        def reset(self, **kwargs):
            obs, info = self.env.reset(**kwargs)  # Call the wrapped env
            self._current_episode_reward = 0
            return obs, info

        def step(self, action):
            obs, reward, terminated, truncated, info = self.env.step(action)
            # Scale the reward
            scaled_reward = reward * self.reward_scale
            self._current_episode_reward += scaled_reward

            if terminated or truncated:
                self.episode_rewards.append(self._current_episode_reward)
                self._episode_count += 1
                # Add episode summary to info
                info['episode'] = {
                    'r': self._current_episode_reward,
                    'l': self._episode_count
                }

            return obs, scaled_reward, terminated, truncated, info

        def summary(self):
            if not self.episode_rewards:
                return "No episodes completed yet."
            return (f"Episodes: {len(self.episode_rewards)}  "
                    f"Mean: {np.mean(self.episode_rewards):.1f}  "
                    f"Std: {np.std(self.episode_rewards):.1f}  "
                    f"Max: {max(self.episode_rewards):.0f}")

    # Use the wrapper
    base_env = gym.make('CartPole-v1')
    wrapped_env = RewardTrackingWrapper(base_env, reward_scale=0.1)  # Scale rewards to 0-1

    # Run 10 episodes
    for ep in range(10):
        obs, _ = wrapped_env.reset(seed=ep)
        done = False
        while not done:
            action = wrapped_env.action_space.sample()
            obs, reward, terminated, truncated, info = wrapped_env.step(action)
            done = terminated or truncated

    wrapped_env.close()
    print("Custom Wrapper Results:")
    print(wrapped_env.summary())
    print("\n(Rewards are scaled by 0.1 — so max episode is now 50 instead of 500)")

else:
    print("Simulated Wrapper Output:")
    print("Episodes: 10  Mean: 2.3  Std: 0.8  Max: 3.6")
    print("(Rewards are scaled by 0.1 — so max episode is now 50 instead of 500)")
    print()
    print("Why scale rewards? Neural networks train better when values are small (~0-1).")
    print("If rewards are huge (e.g., Atari score 10000), gradients can explode.")

## 7. Building a Custom Environment from Scratch

This is the most powerful skill — creating your **own game or simulation** that RL agents can learn to solve.

Any custom environment must implement:
1. `__init__(self)`: define observation_space, action_space
2. `reset(self, seed, options)`: return (initial_observation, info_dict)
3. `step(self, action)`: return (observation, reward, terminated, truncated, info)
4. `render(self)` *(optional)*: visualize the environment

We'll build a **Frozen Lake–style Grid World**:
- 5×5 grid
- Agent starts at (0,0), goal at (4,4)
- Some cells are holes (agent falls in = -10 reward, episode ends)
- Each step: -0.1 penalty (encourages shortest path)
- Reaching goal: +10 reward

In [ ]:
# ── Custom Grid World Environment ─────────────────────────────────────

if GYM_AVAILABLE:
    import gymnasium.spaces as spaces

    class GridWorldEnv(gym.Env):
        """
        A 5x5 grid world environment.

        Think of it like a board game:
        - Agent (robot) starts at top-left
        - Goal (star) is at bottom-right
        - Some squares are holes (black holes = instant death)
        - Find the shortest safe path to the goal!

        Grid Layout:
            S  .  .  H  .
            .  H  .  .  .
            .  .  .  H  .
            .  H  .  .  .
            .  .  .  .  G

        S=Start, G=Goal, H=Hole, .=Empty
        """

        metadata = {'render_modes': ['human', 'rgb_array'], 'render_fps': 4}

        def __init__(self, size=5, render_mode=None):
            super().__init__()
            self.size = size
            self.render_mode = render_mode

            # Fixed hole positions (row, col)
            self.holes = {(0, 3), (1, 1), (2, 3), (3, 1)}
            self.start_pos = np.array([0, 0])
            self.goal_pos = np.array([size-1, size-1])

            # ── Spaces ──────────────────────────────────────────────
            # Observation: agent's (row, col) position
            # Could also use a flattened integer: row * size + col
            self.observation_space = spaces.Box(
                low=0, high=size-1,
                shape=(2,),
                dtype=np.int32
            )

            # Actions: 0=Up, 1=Down, 2=Left, 3=Right
            self.action_space = spaces.Discrete(4)
            self._action_to_direction = {
                0: np.array([-1, 0]),  # Up (decrease row)
                1: np.array([1, 0]),   # Down (increase row)
                2: np.array([0, -1]),  # Left (decrease col)
                3: np.array([0, 1]),   # Right (increase col)
            }
            self.action_names = ['Up', 'Down', 'Left', 'Right']

        def reset(self, seed=None, options=None):
            super().reset(seed=seed)  # Seeds the RNG
            self.agent_pos = self.start_pos.copy()
            self.steps_taken = 0
            observation = self.agent_pos.copy()
            info = {'step': 0}
            return observation, info

        def step(self, action):
            assert self.action_space.contains(action), f"Invalid action {action}"

            # Move agent
            direction = self._action_to_direction[action]
            new_pos = self.agent_pos + direction

            # Clamp to grid boundaries (walls stop movement)
            new_pos = np.clip(new_pos, 0, self.size - 1)
            self.agent_pos = new_pos
            self.steps_taken += 1

            # Check outcomes
            pos_tuple = tuple(self.agent_pos)

            if pos_tuple in self.holes:
                # Fell in a hole!
                reward = -10.0
                terminated = True
                info = {'outcome': 'hole', 'steps': self.steps_taken}

            elif np.array_equal(self.agent_pos, self.goal_pos):
                # Reached the goal!
                reward = +10.0
                terminated = True
                info = {'outcome': 'goal', 'steps': self.steps_taken}

            else:
                # Still walking
                reward = -0.1  # Small time penalty
                terminated = False
                info = {'outcome': 'step', 'steps': self.steps_taken}

            # Truncate after 50 steps to prevent infinite episodes
            truncated = self.steps_taken >= 50

            return self.agent_pos.copy(), reward, terminated, truncated, info

        def render(self, mode='rgb_array'):
            """Return an image of the current grid state."""
            grid = np.zeros((self.size, self.size), dtype=int)
            # 0=empty, 1=hole, 2=goal, 3=agent
            for (r, c) in self.holes:
                grid[r, c] = 1
            grid[self.goal_pos[0], self.goal_pos[1]] = 2
            grid[self.agent_pos[0], self.agent_pos[1]] = 3
            return grid


    # Test the custom environment
    print("Testing Custom GridWorld Environment")
    print("='*40)")

    env = GridWorldEnv(size=5)

    # Verify spaces
    print(f"Observation space: {env.observation_space}")
    print(f"Action space:      {env.action_space}")
    print(f"Actions: 0=Up, 1=Down, 2=Left, 3=Right")
    print()

    obs, info = env.reset(seed=0)
    print(f"Start position: {obs}")
    print(f"Goal position:  {env.goal_pos}")
    print()

    # Try a manual path
    manual_actions = [1,1,3,1,3,3,1,3]  # Down,Down,Right,Down,Right,Right,Down,Right
    total_r = 0
    for i, act in enumerate(manual_actions):
        obs, reward, terminated, truncated, info = env.step(act)
        total_r += reward
        print(f"Step {i+1}: Action={env.action_names[act]:5s} → Pos={obs}, Reward={reward:+.1f}, Done={terminated}")
        if terminated or truncated:
            break

    print(f"\nTotal reward: {total_r:.1f}")
    env.close()

else:
    print("Simulated GridWorld Environment Output:")
    print("Observation space: Box(0, 4, (2,), int32)")
    print("Action space:      Discrete(4) — 0=Up, 1=Down, 2=Left, 3=Right")
    print("Start position: [0 0]")
    print("Goal position:  [4 4]")
    print()
    manual = [(1,'Down',[1,0],-.1,False),(1,'Down',[2,0],-.1,False),(3,'Right',[2,1],-.1,False),
              (1,'Down',[3,0],-.1,False),(3,'Right',[3,1],-10.0,True)]
    total = 0
    for i,(act,name,pos,rew,done) in enumerate(manual):
        total+=rew
        print(f"Step {i+1}: Action={name:5s} → Pos={pos}, Reward={rew:+.1f}, Done={done}")
        if done: break
    print(f"\nTotal reward: {total:.1f}  (fell in a hole!)")

In [ ]:
# ── Visualize the Grid World ──────────────────────────────────────────

def visualize_gridworld(size=5, holes=None, agent_pos=None, goal_pos=None, title='Grid World'):
    """Visualize the grid world environment."""
    if holes is None:
        holes = {(0,3),(1,1),(2,3),(3,1)}
    if agent_pos is None:
        agent_pos = [0, 0]
    if goal_pos is None:
        goal_pos = [size-1, size-1]

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))

    # Draw grid cells
    for r in range(size):
        for c in range(size):
            # Color coding
            if (r, c) in holes:
                color = '#2c2c54'  # Dark purple = hole
                text = '💀' if False else 'H'  # Fall without emoji
            elif [r, c] == goal_pos:
                color = '#ffd32a'  # Yellow = goal
                text = 'G'
            elif [r, c] == agent_pos:
                color = '#0be881'  # Green = agent
                text = 'A'
            else:
                color = '#f8f8f8'  # Light gray = empty
                text = ''

            rect = patches.Rectangle((c, size-1-r), 1, 1, linewidth=2,
                                      edgecolor='#333', facecolor=color)
            ax.add_patch(rect)
            if text:
                ax.text(c + 0.5, size - 1 - r + 0.5, text,
                        ha='center', va='center', fontsize=18, fontweight='bold')

    # Draw optimal path (manual)
    safe_path = [(0,0),(1,0),(2,0),(2,1),(2,2),(3,2),(4,2),(4,3),(4,4)]
    for i in range(len(safe_path)-1):
        r1, c1 = safe_path[i]
        r2, c2 = safe_path[i+1]
        ax.annotate('', xy=(c2+0.5, size-1-r2+0.5), xytext=(c1+0.5, size-1-r1+0.5),
                    arrowprops=dict(arrowstyle='->', color='royalblue', lw=2))

    ax.set_xlim(0, size)
    ax.set_ylim(0, size)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.axis('off')

    # Legend
    legend_elements = [
        patches.Patch(facecolor='#0be881', edgecolor='black', label='A = Agent (start)'),
        patches.Patch(facecolor='#ffd32a', edgecolor='black', label='G = Goal (+10 reward)'),
        patches.Patch(facecolor='#2c2c54', edgecolor='black', label='H = Hole (-10 reward)'),
        patches.Patch(facecolor='#f8f8f8', edgecolor='#333', label='. = Safe cell (-0.1 per step)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.45, 1.0))

    plt.tight_layout()
    plt.savefig('/tmp/gymnasium_gridworld.png', dpi=100, bbox_inches='tight')
    plt.show()

visualize_gridworld(title='Grid World: Find the Path from A to G\n(Blue arrows = one safe path)')

print("Grid World Rules:")
print("  Agent (A) starts at top-left [0,0]")
print("  Goal (G) is at bottom-right [4,4]")
print("  H = Hole: touch it and episode ends with -10")
print("  Each step costs -0.1 (encourages shortest path)")
print("  Reaching goal gives +10")
print("  An RL agent must DISCOVER the safe path through experience!")

## 8. Simple Q-Learning Agent on Grid World

Let's implement **Q-Learning** — the simplest RL algorithm — to solve our Grid World. This shows how an agent learns from scratch.

### The Q-Table Idea

Q-Learning maintains a table Q[state][action] = "how good is this action in this state?".

Analogy: Imagine a book listing every street corner in a city and rating how good it is to turn left/right/go straight/go back from that corner. Q-Learning fills in this book through experience.

**Bellman Equation (the heart of Q-Learning):**
```
Q(s, a) ← Q(s, a) + α × [r + γ × max(Q(s', a')) - Q(s, a)]
              ↑                  ↑           ↑
           old value         reward    best future value
           (α = learning rate, γ = discount factor)
```

- **α (alpha)** = learning rate (0.1 = learn slowly, 1.0 = replace old knowledge entirely)
- **γ (gamma)** = discount factor (0.9 = near future matters, 0.1 = only immediate reward)
- **ε (epsilon)** = exploration rate (explore randomly vs exploit known-good actions)

In [ ]:
# ── Q-Learning on Grid World ──────────────────────────────────────────

class QLearningAgent:
    """
    Tabular Q-Learning agent.
    Works on small, discrete state spaces (like our 5x5 grid).
    For large/continuous spaces, we need Deep Q-Networks (DQN) instead.
    """
    def __init__(self, n_states, n_actions, learning_rate=0.1, gamma=0.95, epsilon=1.0):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate      # α: how fast to update Q-values
        self.gamma = gamma           # γ: how much to value future rewards
        self.epsilon = epsilon       # ε: exploration rate (starts high)
        self.epsilon_min = 0.05      # Never go below 5% random
        self.epsilon_decay = 0.995   # Decay exploration over time

        # Q-table: initialized to zeros
        # Q[state_index][action] = expected future reward
        self.Q = np.zeros((n_states, n_actions))

    def obs_to_state(self, obs, size=5):
        """Convert (row, col) position to flat state index."""
        return int(obs[0]) * size + int(obs[1])

    def choose_action(self, state_idx):
        """ε-greedy: explore randomly OR exploit best known action."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)  # Explore
        else:
            return np.argmax(self.Q[state_idx])        # Exploit

    def update(self, state, action, reward, next_state, done):
        """Bellman equation update."""
        # Estimate of future value
        if done:
            target = reward  # No future if episode ended
        else:
            target = reward + self.gamma * np.max(self.Q[next_state])

        # Update Q-value
        self.Q[state, action] += self.lr * (target - self.Q[state, action])

    def decay_epsilon(self):
        """Reduce exploration over time as agent learns."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


# ── Training ─────────────────────────────────────────────────────────

GRID_SIZE = 5
N_STATES = GRID_SIZE * GRID_SIZE   # 25 states
N_ACTIONS = 4                       # Up, Down, Left, Right
N_EPISODES = 1000

agent = QLearningAgent(N_STATES, N_ACTIONS, learning_rate=0.1, gamma=0.95, epsilon=1.0)

episode_rewards = []
episode_lengths = []
success_history = []  # Did agent reach goal?

if GYM_AVAILABLE:
    env = GridWorldEnv(size=GRID_SIZE)

    for episode in range(N_EPISODES):
        obs, _ = env.reset(seed=episode % 100)
        state = agent.obs_to_state(obs)
        total_reward = 0
        success = False

        for step in range(50):
            action = agent.choose_action(state)
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_state = agent.obs_to_state(next_obs)

            # Learn from this experience
            agent.update(state, action, reward, next_state, terminated or truncated)

            state = next_state
            total_reward += reward

            if info.get('outcome') == 'goal':
                success = True

            if terminated or truncated:
                break

        agent.decay_epsilon()
        episode_rewards.append(total_reward)
        episode_lengths.append(step + 1)
        success_history.append(success)

    env.close()

else:
    # Simulate training curve
    for ep in range(N_EPISODES):
        progress = ep / N_EPISODES
        if progress < 0.2:
            r = np.random.uniform(-10, -5)
            s = False
        elif progress < 0.5:
            r = np.random.uniform(-5, 2)
            s = np.random.random() < 0.3
        else:
            r = np.random.uniform(5, 9)
            s = np.random.random() < 0.85
        episode_rewards.append(r)
        episode_lengths.append(max(5, int(30 * (1 - progress))))
        success_history.append(s)

# ── Plot Training Curves ──────────────────────────────────────────────
window = 50
rewards_smooth = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
success_rate = np.convolve([float(s) for s in success_history], np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reward curve
axes[0].plot(episode_rewards, alpha=0.2, color='steelblue')
axes[0].plot(range(window-1, N_EPISODES), rewards_smooth, color='steelblue', linewidth=2)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Episode Reward (Q-Learning)', fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].grid(True, alpha=0.3)

# Success rate
axes[1].plot(range(window-1, N_EPISODES), success_rate * 100, color='green', linewidth=2)
axes[1].fill_between(range(window-1, N_EPISODES), success_rate * 100, alpha=0.3, color='green')
axes[1].set_title('Goal Reach Rate (rolling 50)', fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Success Rate (%)')
axes[1].set_ylim(0, 105)
axes[1].grid(True, alpha=0.3)

# Q-table heatmap for a slice of actions
Q_max = np.max(agent.Q, axis=1).reshape(GRID_SIZE, GRID_SIZE)
im = axes[2].imshow(Q_max, cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=axes[2])
axes[2].set_title('Q-Table: Max Q-Value per State', fontweight='bold')
axes[2].set_xlabel('Column')
axes[2].set_ylabel('Row')
# Mark holes and goal
for (r, c) in [(0,3),(1,1),(2,3),(3,1)]:
    axes[2].add_patch(patches.Rectangle((c-0.5, r-0.5), 1, 1, fill=False, edgecolor='black', linewidth=3))
axes[2].plot(4, 4, 'y*', markersize=15, label='Goal')
axes[2].legend()

plt.suptitle('Q-Learning Training on Grid World', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/gymnasium_qlearning.png', dpi=100, bbox_inches='tight')
plt.show()

final_success = np.mean(success_history[-100:]) * 100
print(f"\nTraining complete!")
print(f"Final 100-episode success rate: {final_success:.1f}%")
print(f"Final exploration rate (ε): {agent.epsilon:.3f}")
print(f"\nThe agent learned to navigate the grid without being told the rules!")

## 9. Common Pitfalls & How to Avoid Them

In [ ]:
# ── Common Pitfalls ───────────────────────────────────────────────────

print("="*65)
print(" Common Gymnasium Pitfalls & Their Fixes")
print("="*65)

pitfalls = [
    {
        "title": "1. Forgetting to call env.reset() at episode start",
        "wrong": "obs = env.observation_space.sample()  # Wrong! Not the actual state",
        "right": "obs, info = env.reset()  # Always reset to get initial observation",
        "why": "env.observation_space.sample() returns a random obs, not the real start state."
    },
    {
        "title": "2. Not checking for episode end (terminated OR truncated)",
        "wrong": "while not terminated:  # Missing truncated!",
        "right": "while not (terminated or truncated):",
        "why": "If you only check terminated, time-limited episodes run forever."
    },
    {
        "title": "3. Forgetting env.close() after training",
        "wrong": "# Just let the env go out of scope",
        "right": "env.close()  # Release resources, close windows",
        "why": "Can cause memory leaks or zombie processes, especially with rendering."
    },
    {
        "title": "4. Wrong Gym vs Gymnasium import",
        "wrong": "import gym  # Old API: step() returns 4 values",
        "right": "import gymnasium as gym  # New API: step() returns 5 values",
        "why": "Old gym returns (obs, reward, done, info). New gymnasium returns (obs, reward, terminated, truncated, info)."
    },
    {
        "title": "5. Not resetting between episodes during evaluation",
        "wrong": "# Running multiple episodes without reset",
        "right": "for ep in range(n): obs, _ = env.reset(); ...",
        "why": "State carries over between episodes. Always reset at episode start."
    },
    {
        "title": "6. Using gym.make without setting render_mode for video",
        "wrong": "env = gym.make('CartPole-v1'); env.render()  # Returns None",
        "right": "env = gym.make('CartPole-v1', render_mode='rgb_array'); frame = env.render()",
        "why": "render() only works if render_mode is specified at gym.make() time."
    },
]

for p in pitfalls:
    print(f"\n{'─'*65}")
    print(f"  {p['title']}")
    print(f"  WHY: {p['why']}")
    print(f"  ✗ Wrong: {p['wrong']}")
    print(f"  ✓ Right: {p['right']}")

print(f"\n{'='*65}")

## 10. Mini Project: Complete Training Pipeline with Visualization

In [ ]:
# ── Mini Project: Visualize the Learned Policy ────────────────────────
#
# After training, we can extract what the agent learned:
# For every state (grid cell), what action does the Q-table recommend?
# This is called the POLICY: π(s) = argmax_a Q(s, a)

def visualize_policy(Q_table, size=5, holes=None, goal_pos=None):
    """Show what action the agent takes at each grid cell."""
    if holes is None:
        holes = {(0,3),(1,1),(2,3),(3,1)}
    if goal_pos is None:
        goal_pos = [size-1, size-1]

    action_symbols = ['↑', '↓', '←', '→']  # Up, Down, Left, Right
    action_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: Policy (best action per state)
    ax = axes[0]
    for r in range(size):
        for c in range(size):
            state_idx = r * size + c
            pos_tuple = (r, c)

            if pos_tuple in holes:
                rect = patches.Rectangle((c, size-1-r), 1, 1, facecolor='#2c2c54', edgecolor='#333', linewidth=2)
                ax.add_patch(rect)
                ax.text(c+0.5, size-1-r+0.5, 'H', ha='center', va='center', fontsize=16, color='white', fontweight='bold')
            elif [r, c] == goal_pos:
                rect = patches.Rectangle((c, size-1-r), 1, 1, facecolor='#ffd32a', edgecolor='#333', linewidth=2)
                ax.add_patch(rect)
                ax.text(c+0.5, size-1-r+0.5, 'G', ha='center', va='center', fontsize=16, fontweight='bold')
            else:
                best_action = np.argmax(Q_table[state_idx])
                q_val = np.max(Q_table[state_idx])
                # Color by Q-value strength
                intensity = max(0, min(1, (q_val + 2) / 12))  # normalize
                color = plt.cm.YlGn(intensity)
                rect = patches.Rectangle((c, size-1-r), 1, 1, facecolor=color, edgecolor='#333', linewidth=2)
                ax.add_patch(rect)
                ax.text(c+0.5, size-1-r+0.5, action_symbols[best_action],
                        ha='center', va='center', fontsize=20, color='#2c3e50', fontweight='bold')
                ax.text(c+0.5, size-1-r+0.15, f'{q_val:.1f}',
                        ha='center', va='center', fontsize=8, color='#555')

    ax.set_xlim(0, size)
    ax.set_ylim(0, size)
    ax.set_aspect('equal')
    ax.set_title('Learned Policy (arrows = best action)\nNumbers = Q-value', fontsize=12, fontweight='bold')
    ax.axis('off')

    # Right: Q-value heatmap per action
    ax2 = axes[1]
    Q_max = np.max(Q_table, axis=1).reshape(size, size)
    im = ax2.imshow(Q_max, cmap='RdYlGn', aspect='auto', vmin=-10, vmax=10)
    plt.colorbar(im, ax=ax2, label='Max Q-value')

    for r in range(size):
        for c in range(size):
            ax2.text(c, r, f'{Q_max[r,c]:.1f}', ha='center', va='center',
                    fontsize=10, color='black', fontweight='bold')

    ax2.set_title('Q-Value Heatmap\n(Green=High Value, Red=Low Value)', fontsize=12, fontweight='bold')
    ax2.set_xticks(range(size))
    ax2.set_yticks(range(size))
    ax2.set_xlabel('Column')
    ax2.set_ylabel('Row')

    plt.suptitle('What the Q-Learning Agent Learned', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/tmp/gymnasium_policy.png', dpi=100, bbox_inches='tight')
    plt.show()


visualize_policy(agent.Q, size=GRID_SIZE)

print("Reading the policy visualization:")
print("  ↑↓←→ = The action the agent takes at that cell")
print("  Numbers below arrows = Q-value (expected future reward)")
print("  Green cells = high value (on the path to goal)")
print("  Red cells = low value (dead ends, near holes)")
print("  H cells = holes (avoid!)")
print("  G = goal (highest value!)")

## 11. Interview Q&A

---

### Q1: What is the difference between Gymnasium and OpenAI Gym?
**A**: OpenAI Gym (the original) is no longer maintained by OpenAI. Gymnasium is its maintained fork by the Farama Foundation. Key API change: `env.step()` now returns 5 values `(obs, reward, terminated, truncated, info)` instead of 4 `(obs, reward, done, info)`. The `done` flag was split into `terminated` (natural end — goal/failure) and `truncated` (time limit hit). This is cleaner because an agent should behave differently when it ran out of time vs. when it actually fell off a cliff.

---

### Q2: What is the difference between `Discrete` and `Box` action spaces?
**A**: `Discrete(n)` is for categorical choices — pick one of n options (like a menu). Example: CartPole has Discrete(2) — push left or push right. `Box(low, high, shape)` is for continuous control — real numbers in a range. Example: A robot arm torque can be any value from -1.0 to +1.0. Q-Learning works with Discrete. For Box spaces, you need policy gradient algorithms like PPO or SAC.

---

### Q3: What is the ε-greedy strategy?
**A**: ε-greedy balances **exploration** (trying new things) vs **exploitation** (using what you know). With probability ε, take a random action (explore). With probability 1-ε, take the best known action (exploit). ε starts high (like 1.0 = fully random) and decays over time as the agent learns. If ε is always 0, the agent gets stuck on the first "good enough" solution and never discovers better ones. If ε stays at 1.0, it never uses what it learned.

---

### Q4: What is the discount factor γ (gamma)?
**A**: γ controls how much the agent cares about **future rewards** vs immediate rewards. γ=1.0 means future rewards are as valuable as immediate ones (far-sighted). γ=0.0 means only care about the immediate reward (completely shortsighted). In practice, γ=0.95 or 0.99 is common. Why discount at all? Because future rewards are uncertain — a reward you get now is guaranteed; a reward 10 steps from now might never happen if the episode ends early.

---

### Q5: When does Q-Learning fail, and what replaces it?
**A**: Q-Learning uses a table Q[state][action]. This only works when the state space is small and discrete. For CartPole (4 floats), there are infinitely many states — you can't make a table. The solution is **Deep Q-Networks (DQN)**: replace the table with a neural network that takes the observation and outputs Q-values for all actions. DQN is Q-Learning + neural network approximation.

---

### Q6: What is the exploration-exploitation dilemma?
**A**: The classic RL dilemma. Exploitation = use what you already know to get the best reward you know of. Exploration = try new actions to find potentially better rewards. Too much exploitation: you get stuck in local optima (the first decent solution). Too much exploration: you never converge on a good policy. The balance matters — early in training, explore more; later, exploit more. This is exactly what ε-decay does.

---

### Q7: How do you design a good reward function?
**A**: Reward design is one of the hardest parts of RL. Key principles:
1. **Sparse vs dense rewards**: Sparse (only at goal) is harder to learn from; dense (shaping rewards along the way) helps but can cause unintended behavior.
2. **Reward hacking**: Agents find shortcuts. If you reward "time alive" in a boat game, the boat rocks in place forever.
3. **Don't reward the proxy, reward the goal**: Reward the actual goal, not what you think leads to the goal.
4. **Scale matters**: Keep rewards in a reasonable range (e.g., [-1, +1]) for stable training.

## 12. Resources

### Official Documentation
- **Gymnasium Docs**: https://gymnasium.farama.org/
- **Gymnasium GitHub**: https://github.com/Farama-Foundation/Gymnasium
- **Environment List**: https://gymnasium.farama.org/environments/

### Video Tutorials
- **Nicholas Renotte — RL Course**: https://www.youtube.com/watch?v=bD6V3rcr_54
- **Phil Tabor — Q-Learning Tutorial**: https://www.youtube.com/watch?v=5fHngyN8Qhw
- **Two Minute Papers — RL highlights**: https://www.youtube.com/channel/UCbfYPyITQ-7l4upoX8nvctg

### Courses & Books
- **Sutton & Barto (FREE)** — The RL Bible: http://incompleteideas.net/book/the-book-2nd.html
- **Spinning Up in RL (OpenAI)**: https://spinningup.openai.com/en/latest/
- **Hugging Face Deep RL Course (FREE)**: https://huggingface.co/learn/deep-rl-course/
- **DeepMind + UCL Lectures**: https://www.youtube.com/playlist?list=PLqYmG7hTraZBKeNJ-JE_eyJHZ7XgBoAyb

### Research Papers
- **DQN (Playing Atari with Deep RL)**: https://arxiv.org/abs/1312.5602 — The paper that started deep RL
- **PPO (Proximal Policy Optimization)**: https://arxiv.org/abs/1707.06347 — Most popular RL algorithm
- **AlphaGo**: https://www.nature.com/articles/nature16961 — Beat world Go champion

---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| RL Loop | `reset()` → `step(action)` → repeat until done |
| `step()` returns | `(obs, reward, terminated, truncated, info)` |
| Observation space | What the agent sees (shape of its input) |
| Action space | What the agent can do (Discrete or Box) |
| Q-Learning | Tabular method; works for small discrete spaces |
| ε-greedy | Explore randomly vs exploit learned knowledge |
| Custom env | Inherit `gym.Env`, implement `reset()` and `step()` |
| Wrappers | Modify env behavior without touching source code |

**Next**: Stable-Baselines3 — use PPO, DQN, SAC algorithms with 3 lines of code!

```python
# The promise of SB3:
from stable_baselines3 import PPO
model = PPO('MlpPolicy', env)
model.learn(total_timesteps=100_000)
# Done! Agent trained with state-of-the-art PPO algorithm.
```